In [3]:
import json
from PIL import Image, ImageDraw, ImageFont
import subprocess
import sys

import pymongo
import os

sys.path.append(os.path.abspath(".."))
from core.utilities import draw_wrapped_text
from PIL import Image, ImageDraw, ImageFont


from dotenv import load_dotenv
load_dotenv(override=True) 

db_client = pymongo.MongoClient(os.environ['REMOTE_MONGO_DB'])
db = db_client['db_movie'] 

In [ ]:
PROJECT_FOLDER = '/Users/sangdo/Downloads/movie-top-list/'
TRAILER_FOLDER = PROJECT_FOLDER + 'trailers/'
BG_FILE = PROJECT_FOLDER + 'timeline_template/timeline_30_items_57600.png'
OUTPUT_LONG_VIDEO_FOLDER = PROJECT_FOLDER + 'long_video/'

first_video_x = 5
firxt_video_y = 5
CANVAS_WIDTH = 38700    #length of background image
CANVAS_HEIGHT = 1080
VIEWPORT_WIDTH = 1920
VIEWPORT_HEIGHT = 1080
VIDEO_DURATION = 480    #4 minutes
BUFFER_END_DURATION = 10    #for the last video to play to the end

CLIP_W = 1280
CLIP_H = 720
#Scrollable distance: 19980 - 1920 = 18060 px
#Desired duration: 480
#Required speed: 18060 / 480 = 37.625 px/sec
SCROLL_SPEED = ((CANVAS_WIDTH - VIEWPORT_WIDTH) / VIDEO_DURATION) * 1  #pixels per sec

CARD_W = 1290
#define styles of texts


<h2>Find list of trailers of the actor<h2>

In [5]:
def get_trailer_ids(actor_name):
    ids = []
    tb_paycheck = db['tb_paycheck']
    data = tb_paycheck.find({'actor': actor_name}).sort({'year':1}) #.limit(4)

    for item in data:
        ids.append(item['trailer'])
    return ids

#
# get_trailer_ids('Tom Cruise')

In [6]:
actor_name = 'Tom Cruise'

<h2>Put a video at position X, Y inside a video</h2>

In [ ]:
def put_video_inside_video(actor_key, trailer_ids):
    inputs = ""
    filters = ""

    # build ffmpeg input list
    i = 0
    for trailer_id in trailer_ids:
        filepath = TRAILER_FOLDER + actor_key + "/"+trailer_id+".mp4"
        inputs += f"-i {filepath} "

        x = first_video_x + i * CLIP_W  #position of the clip
        trigger_x = VIEWPORT_WIDTH * 1 / 2    #start playing while reach to the center of the screen

        delay = max((x - trigger_x) / SCROLL_SPEED, 0)
        play_time = 20

        filters += (
            f"[{i+1}:v]"
            f"scale={CLIP_W}:{CLIP_H},"
            f"trim=duration={play_time},"
            f"tpad=start_duration={delay}:start_mode=clone,"
            f"tpad=stop_duration={VIDEO_DURATION + BUFFER_END_DURATION}:stop_mode=clone"
            f"[v{i}];"
        )

        i = i + 1

    # create base canvas
    filters += f"[0:v]scale={CANVAS_WIDTH}:{CANVAS_HEIGHT}[base];"

    last = "base"

    # overlay clips in position
    i = 0
    for trailer_id in trailer_ids:
        x = CARD_W * i + first_video_x
        filters += f"[{last}][v{i}]overlay={x}:{firxt_video_y}:eof_action=pass[tmp{i}];"
        last = f"tmp{i}"
        i = i + 1

    # scrolling camera
    filters += f"[{last}]crop={VIEWPORT_WIDTH}:{VIEWPORT_HEIGHT}:x='t*{SCROLL_SPEED}':y=0[out]"  #standard youtube landscape video size
    output_filename = OUTPUT_LONG_VIDEO_FOLDER + actor_key + ".mp4"
    cmd = f'ffmpeg -loglevel error -y -loop 1 -i {BG_FILE} {inputs} -filter_complex "{filters}" -map "[out]" -t {VIDEO_DURATION + BUFFER_END_DURATION} -r 30 -c:v libx264 -preset ultrafast -crf 23 -movflags +faststart -pix_fmt yuv420p {output_filename}'
    print("Running FFmpeg...")
    subprocess.run(cmd, shell=True)

#test
trailer_ids = get_trailer_ids(actor_name)
# put_video_inside_video('tom_cruise', trailer_ids)    #size w 57600: 8 minutes ~ 4 minutes to produce

Running FFmpeg...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, png_pipe, from '/Users/sangdo/Downloads/movie-top-list/timeline_template/timeline_30_items.png':
  Duration: N/A, bitrate: N/A
  Stream #0:0: Video: png, rgba(pc, gbr/unknown/unknown), 19980x10

In [8]:
FONT_PATH = "font/Inter.ttf"

CARD_DETAILS = {
    'title': {
        'position': (35, 475),
        'font_size': 50,
        'font_color': 'white'
    },
    'budget': {
        'position': (40, 660),
        'font_size': 50,
        'font_color': 'white'
    },
    'year': { #revenue
        'position': (295, 650),
        'font_size': 50,
        'font_color': 'white'   #004aad
    },
    'box_office': {
        'position': (470, 660),
        'font_size': 50,
        'font_color': 'white'
    },
    'paycheck': {
        'position': (370, 755),
        'font_size': 50,
        'font_color': 'white'   #5d18eb
    },
    'coin': {
        'position': (35, 475),
        'font_size': 50,
        'font_color': 'white'
    }
}
#clone new background image for the video
def create_bg_image():
    # tb_paycheck = db['tb_paycheck']
    # data = tb_paycheck.find({'actor': actor_name}).sort({'year':1}) #.limit(4)
    #put data into each card
    base = Image.open(PROJECT_FOLDER + 'timeline_template/empty_card_666x1080.png').convert("RGBA")
    draw = ImageDraw.Draw(base)
    #draw salary range
    draw_wrapped_text(
        draw=draw,
        position= CARD_DETAILS['title']['position'],
        text= 'abc',
        font= ImageFont.truetype(FONT_PATH, CARD_DETAILS['title']['font_size']),
        max_width= 600,
        fill= (255, 255, 255)   #text color
    )

    base.convert("RGB").save("new_card.png", "PNG")
###
# create_bg_image()